# PhiUSIIL Phishing URL Detection: XGBoost vs BERT-base
Trains the two strongest models from the paper on the PhiUSIIL dataset and reports test accuracy.

**Tip:** run on a GPU runtime (e.g. Colab T4) so BERT trains in reasonable time.

In [3]:
!pip install -q ucimlrepo xgboost transformers accelerate scikit-learn

## 1. Load dataset

In [4]:
import numpy as np, pandas as pd, torch, time
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

data = fetch_ucirepo(id=967)
X = data.data.features
y = data.data.targets['label'].values
df = X.copy()
df['label'] = y
print(df.shape)
print(df['label'].value_counts())

(235795, 55)
label
1    134850
0    100945
Name: count, dtype: int64


## 2. Train / test split (80/20, stratified)

In [5]:
idx = np.arange(len(df))
train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)
results = {}

def report(name, y_true, y_pred):
    results[name] = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
    }
    print(name, {k: round(v, 4) for k, v in results[name].items()})

## 3. Model 1: XGBoost (tabular features)

In [4]:
from xgboost import XGBClassifier

X_num = X.select_dtypes(include=[np.number])   # drops URL / Domain / TLD / Title text columns
print('Numeric features used:', X_num.shape[1])

xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    tree_method='hist', random_state=42, n_jobs=-1)
xgb.fit(X_num.iloc[train_idx], y[train_idx])
report('XGBoost', y[test_idx], xgb.predict(X_num.iloc[test_idx]))

Numeric features used: 50
XGBoost {'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0}


## 4. Model 2: BERT-base (raw URL text)

In [5]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

urls = df['URL'].astype(str).values
tok = AutoTokenizer.from_pretrained('bert-base-uncased')

class URLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=64)
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = int(self.labels[i])
        return item

train_ds = URLDataset(urls[train_idx], y[train_idx])
test_ds  = URLDataset(urls[test_idx],  y[test_idx])

bert = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

args = TrainingArguments(
    output_dir='bert_out', num_train_epochs=1, learning_rate=2e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=128,
    fp16=torch.cuda.is_available(), save_strategy='no', report_to='none', seed=42)

trainer = Trainer(model=bert, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorWithPadding(tok))
trainer.train()

pred = trainer.predict(test_ds).predictions.argmax(axis=1)
report('BERT-base', y[test_idx], pred)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.033480
1000,0.011848
1500,0.016073
2000,0.012714
2500,0.012987
3000,0.008990
3500,0.013761
4000,0.011779
4500,0.007639
5000,0.011395


BERT-base {'Accuracy': 0.9982, 'Precision': 0.9968, 'Recall': 1.0, 'F1': 0.9984}


## 5. Results

In [6]:
summary = pd.DataFrame(results).T
summary['Accuracy (%)'] = (summary['Accuracy'] * 100).round(2)
summary

,Accuracy,Precision,Recall,F1,Accuracy (%)
XGBoost,1.000000,1.000000,1.0,1.00000,100.00
BERT-base,0.998155,0.996785,1.0,0.99839,99.82


In [9]:
!pip install -q ucimlrepo

import re, unicodedata
import pandas as pd
from urllib.parse import urlsplit, urlunsplit
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score

# 1. Load and flip labels (1 = phishing)
data = fetch_ucirepo(id=967)
df = data.data.features[['URL']].copy()
df['label'] = 1 - data.data.targets['label'].values

def normalize(url):
    url = unicodedata.normalize('NFKC', str(url).strip())
    url = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', url)
    if '://' not in url:
        url = 'http://' + url
    try:
        p = urlsplit(url)
        host = (p.hostname or '').lower().encode('idna').decode('ascii')
        if p.port:
            host += ':' + str(p.port)
        return urlunsplit((p.scheme.lower(), host, p.path, p.query, p.fragment))
    except Exception:
        return url

df['URL'] = df['URL'].map(normalize)
df = df.drop_duplicates(subset='URL').reset_index(drop=True)
df['host'] = df['URL'].map(lambda u: urlsplit(u).hostname or '')
print("Phishing share:", df['label'].mean())   # should be ~0.427

# 2. Same host-disjoint split as your notebook
gss = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=1337)
train_idx, rest_idx = next(gss.split(df, groups=df['host']))
rest = df.iloc[rest_idx]
gss = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=1337)
val_i, test_i = next(gss.split(rest, groups=rest['host']))
train, val, test = df.iloc[train_idx], rest.iloc[val_i], rest.iloc[test_i]
print(len(train), len(val), len(test))         # should be 164901 36103 34354

# 3. Template check and rule baseline
tmpl = df['URL'].str.match(r'^https://www\.[^/?#]+/?$')
print(pd.crosstab(df['label'], tmpl, normalize='index'))

rule_pred = (~tmpl).astype(int)   # predict phishing if URL does NOT match template
t = test.index
print("Rule accuracy:", accuracy_score(test['label'], rule_pred.loc[t]))
print("Rule F1:      ", f1_score(test['label'], rule_pred.loc[t]))

Phishing share: 0.4270430578098046
164901 36103 34354
URL       False     True 
label                    
0      0.000000  1.000000
1      0.989682  0.010318
Rule accuracy: 0.9957792396809687
Rule F1:       0.9948271556491027


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def host_only(u):
    h = urlsplit(u).hostname or ''
    return h[4:] if h.startswith('www.') else h

def run(col_train, col_test, name):
    vec = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3)
    Xtr, Xte = vec.fit_transform(col_train), vec.transform(col_test)
    m = LogisticRegression(C=1.0, max_iter=1000).fit(Xtr, train['label'])
    p = m.predict(Xte); s = m.predict_proba(Xte)[:, 1]
    hard = (test['label'] == 1) & tmpl.loc[test.index]      # phishing that looks like the template
    print(f"{name:28s} acc={accuracy_score(test['label'], p):.4f}  "
          f"F1={f1_score(test['label'], p):.4f}  AUROC={roc_auc_score(test['label'], s):.4f}  "
          f"hard-phish recall={p[hard.values].mean():.4f} (n={hard.sum()})")

run(train['URL'], test['URL'], "LR original URL")
run(train['URL'].map(host_only), test['URL'].map(host_only), "LR host-only (no template)")

LR original URL              acc=0.9957  F1=0.9947  AUROC=0.9988  hard-phish recall=0.2966 (n=145)
LR host-only (no template)   acc=0.8647  F1=0.8147  AUROC=0.9159  hard-phish recall=0.6966 (n=145)


In [11]:
import random
random.seed(1337)

def host_only(u):
    h = urlsplit(u).hostname or ''
    return '.'.join(l for l in h.split('.') if l != 'www')

def fit(col):
    vec = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3)
    m = LogisticRegression(C=1.0, max_iter=1000).fit(vec.fit_transform(col), train['label'])
    return vec, m

vec_o, m_o = fit(train['URL'])
vec_h, m_h = fit(train['URL'].map(host_only))

# Mutation operators following the paper's Section 3.6.1
SUB   = ['secure', 'login', 'verify', 'update', 'auth', 'account', 'billing', 'support']
TLD   = {1: ['com', 'net', 'org', 'co'], 2: ['io', 'info', 'biz', 'site'], 3: ['shop', 'click', 'top', 'xyz']}
GLYPH = {'a':'а','c':'с','e':'е','i':'і','o':'о','p':'р','x':'х','y':'у','s':'ѕ','l':'ӏ'}  # Cyrillic look-alikes

def with_host(u, h):
    p = urlsplit(u)
    return urlunsplit((p.scheme, h, p.path, p.query, p.fragment))

def subdomain(h, d):
    return '.'.join(random.sample(SUB, d)) + '.' + h

def tldswap(h, d):
    labs = h.split('.')
    labs[-1] = random.choice([t for t in TLD[d] if t != labs[-1]])
    return '.'.join(labs)

def typo(h, d):
    labs = h.split('.')
    cand = [k for k in range(len(labs) - 1) if labs[k] != 'www'] or [0]
    i = max(cand, key=lambda k: len(labs[k]))
    s = list(labs[i])
    for _ in range(d):
        if len(s) < 3: break
        j = random.randrange(len(s) - 1)
        op = random.choice(['swap', 'drop', 'dup'])
        if op == 'swap':   s[j], s[j+1] = s[j+1], s[j]
        elif op == 'drop': del s[j]
        else:              s.insert(j, s[j])
    labs[i] = ''.join(s)
    return '.'.join(labs)

def homoglyph(h, d):
    labs, out = h.split('.'), []
    for k, lab in enumerate(labs):
        if k == len(labs) - 1 or lab == 'www':
            out.append(lab); continue
        idx = [j for j, c in enumerate(lab) if c in GLYPH]
        n = max(1, int(len(idx) * 0.2 * d)) if idx else 0
        pick = set(random.sample(idx, n))
        new = ''.join(GLYPH[c] if j in pick else c for j, c in enumerate(lab))
        try:    out.append(new.encode('idna').decode('ascii'))
        except Exception: out.append(lab)
    return '.'.join(out)

# Evaluate both models on clean and mutated test URLs
y = test['label'].values
def acc_pair(urls):
    return (accuracy_score(y, m_o.predict(vec_o.transform(urls))),
            accuracy_score(y, m_h.predict(vec_h.transform(urls.map(host_only)))))

rows = [('clean', 0, *acc_pair(test['URL']))]
for fam, f in [('subdomain', subdomain), ('typo', typo), ('tldswap', tldswap), ('homoglyph', homoglyph)]:
    for d in (1, 2, 3):
        mut = test['URL'].map(lambda u: with_host(u, f(urlsplit(u).hostname or '', d)))
        rows.append((fam, d, *acc_pair(mut)))

res = pd.DataFrame(rows, columns=['family', 'depth', 'LR original', 'LR host-only']).round(4)
print(res.to_string(index=False))

   family  depth  LR original  LR host-only
    clean      0       0.9957        0.8647
subdomain      1       0.4694        0.6585
subdomain      2       0.4140        0.4861
subdomain      3       0.4103        0.4228
     typo      1       0.9952        0.8565
     typo      2       0.9955        0.8496
     typo      3       0.9953        0.8428
  tldswap      1       0.9949        0.8149
  tldswap      2       0.9951        0.6706
  tldswap      3       0.9959        0.4603
homoglyph      1       0.9904        0.4984
homoglyph      2       0.9899        0.4775
homoglyph      3       0.9897        0.4652


In [12]:
# A) Do benign hosts ever have subdomains once 'www' is removed?
nlab = df['URL'].map(lambda u: len(host_only(u).split('.'))).clip(upper=5)
print(pd.crosstab(nlab, df['label'], normalize='columns').round(4), "\n")   # columns: 0=benign, 1=phishing

# B) Per-class results: phishing recall (evasion) and benign accuracy (false positives)
def per_class(urls):
    out = []
    for vec, m, tf in [(vec_o, m_o, lambda s: s), (vec_h, m_h, lambda s: s.map(host_only))]:
        p = m.predict(vec.transform(tf(urls)))
        out += [p[y == 1].mean(), 1 - p[y == 0].mean()]
    return out

random.seed(1337)
rows = [('clean', 0, *per_class(test['URL']))]
for fam, f in [('subdomain', subdomain), ('typo', typo), ('tldswap', tldswap), ('homoglyph', homoglyph)]:
    for d in (1, 2, 3):
        mut = test['URL'].map(lambda u: with_host(u, f(urlsplit(u).hostname or '', d)))
        rows.append((fam, d, *per_class(mut)))

cols = ['family', 'depth', 'orig phish-recall', 'orig benign-acc', 'host phish-recall', 'host benign-acc']
print(pd.DataFrame(rows, columns=cols).round(4).to_string(index=False))

label       0       1
URL                  
2      0.8518  0.4274
3      0.1352  0.4304
4      0.0125  0.1137
5      0.0004  0.0285 

   family  depth  orig phish-recall  orig benign-acc  host phish-recall  host benign-acc
    clean      0             0.9895           1.0000             0.7254           0.9615
subdomain      1             1.0000           0.1005             0.9202           0.4766
subdomain      2             1.0000           0.0066             0.9848           0.1394
subdomain      3             1.0000           0.0003             0.9985           0.0226
     typo      1             0.9883           1.0000             0.7077           0.9600
     typo      2             0.9889           1.0000             0.6924           0.9588
     typo      3             0.9886           1.0000             0.6796           0.9562
  tldswap      1             0.9875           1.0000             0.6142           0.9544
  tldswap      2             0.9880           1.0000             

In [13]:
!pip install -q tldextract
import tldextract
ext = tldextract.TLDExtract()
has_sub = df['URL'].map(lambda u: ext(host_only(u)).subdomain != '')
print(pd.crosstab(df['label'], has_sub, normalize='index').round(4))   # True column = has a real subdomain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 6.3 MB/s eta 0:00:00
URL     False   True 
label                
0      0.9712  0.0288
1      0.4566  0.5434
